# PoC (análisis): corpus y preparación para RAG

Esta notebook sirve para **explorar** el material normativo (PDF / texto ya limpio) y **diseñar** transformaciones basadas en **expresiones regulares** antes de la ingesta al índice RAG.

**Objetivo:** entender estructuras repetibles (casos, reglas, secciones, ruido bibliográfico) para definir segmentación, metadatos y limpieza que luego se implementen en el pipeline. Las métricas agregadas (caracteres, stop-words, gráficos) están en `PoC (metricas).ipynb`.


## Recuperación híbrida (léxica + semántica) y rol de las regex

- **Recuperación léxica (BM25 / similar):** necesita tokens “útiles”; conviene **normalizar** guiones de PDF, espacios y delimitar bloques para que el texto coincida mejor con la consulta.
- **Recuperación semántica (embeddings):** se beneficia de **chunks coherentes** (un caso, una regla + contexto) y de **menos ruido** (códigos bibliográficos al pie que el usuario nunca escribe) para no desperdiciar capacidad del vector.
- **Regex:** sirve para **detectar delimitadores** (`CASE 12`, `Rule 10`), **extraer metadatos** (lista de reglas citadas por caso) y **reglas de limpieza** acotadas (p. ej. referencias `GBR 1962/25` solo al final de página). No reemplaza al modelo semántico, pero alinea corpus y consultas para que **ambos** brazos del híbrido funcionen mejor.


## Configuración: rutas al corpus y texto limpio

La siguiente celda resuelve la raíz del repo y las carpetas `corpus/` (PDF) y, si existe, `corpus_limpio/` generado por la notebook de métricas. Así podés analizar **bruto vs preprocesado**.


In [2]:
from __future__ import annotations

import re
from pathlib import Path

from pypdf import PdfReader

_cwd = Path.cwd().resolve()


def resolver_repo() -> Path:
    if (_cwd / "corpus").is_dir():
        return _cwd
    if (_cwd.parent / "corpus").is_dir():
        return _cwd.parent
    raise FileNotFoundError("No se encontró corpus/; ejecutá desde la raíz del repo o desde notebooks auxiliares/")


def resolver_notebooks_aux() -> Path:
    for name in ("PoC (análisis).ipynb", "PoC (metricas).ipynb", "PoC.ipynb"):
        if (_cwd / name).exists():
            return _cwd
    na = _cwd / "notebooks auxiliares"
    if any((na / n).exists() for n in ("PoC (análisis).ipynb", "PoC (metricas).ipynb")):
        return na
    na2 = _cwd.parent / "notebooks auxiliares"
    if any((na2 / n).exists() for n in ("PoC (análisis).ipynb", "PoC (metricas).ipynb")):
        return na2
    return resolver_repo() / "notebooks auxiliares"


REPO = resolver_repo()
CORPUS_DIR = REPO / "corpus"
NB_AUX = resolver_notebooks_aux()
CORPUS_LIMPIO_DIR = NB_AUX / "corpus_limpio"

print("REPO:", REPO)
print("CORPUS (PDF):", CORPUS_DIR)
print("CORPUS_LIMPIO:", CORPUS_LIMPIO_DIR, "— existe:", CORPUS_LIMPIO_DIR.is_dir())


REPO: /Users/marcelo.luna/Library/CloudStorage/OneDrive-Personal/Diplomaturas/01. Inteligencia Artificial Aplicada/Materias/05. Taller de Trabajo Final/Repositorio/DIIA_trabajo_final
CORPUS (PDF): /Users/marcelo.luna/Library/CloudStorage/OneDrive-Personal/Diplomaturas/01. Inteligencia Artificial Aplicada/Materias/05. Taller de Trabajo Final/Repositorio/DIIA_trabajo_final/corpus
CORPUS_LIMPIO: /Users/marcelo.luna/Library/CloudStorage/OneDrive-Personal/Diplomaturas/01. Inteligencia Artificial Aplicada/Materias/05. Taller de Trabajo Final/Repositorio/DIIA_trabajo_final/notebooks auxiliares/corpus_limpio — existe: True


## Cargar muestras de texto para experimentar

- Si hay archivos `*_limpio.txt`, usamos el **Case Book** como muestra larga (truncada) para no saturar la salida.
- Si no hay limpio, extraemos las **primeras páginas** del PDF del Case Book.

Así las regex siguen aplicándose sobre texto parecido al que usará el RAG tras la ingesta.


In [3]:
MAX_CHARS = 120_000  # tope para análisis interactivo


def muestra_case_book() -> tuple[str, str]:
    """Devuelve (etiqueta_origen, texto)."""
    patron_limpo = "*Case-Book*_limpio.txt"
    if CORPUS_LIMPIO_DIR.is_dir():
        for p in sorted(CORPUS_LIMPIO_DIR.glob("*_limpio.txt")):
            if "Case-Book" in p.name or "WS-Case" in p.name:
                txt = p.read_text(encoding="utf-8")
                return str(p.name), txt[:MAX_CHARS]
    pdf = CORPUS_DIR / "WS-Case-Book-2025-2028-v2025-07.pdf"
    if not pdf.is_file():
        pdf = next(CORPUS_DIR.glob("*Case*Book*.pdf"), None)
    if pdf is None or not pdf.is_file():
        raise FileNotFoundError("No se encontró Case Book en corpus/")
    chunks: list[str] = []
    reader = PdfReader(str(pdf))
    for i, page in enumerate(reader.pages[:25]):
        chunks.append(page.extract_text() or "")
    return pdf.name, "\n".join(chunks)[:MAX_CHARS]


origen, texto_cb = muestra_case_book()
print(f"Muestra: {origen} — {len(texto_cb):,} caracteres")


Muestra: WS-Case-Book-2025-2028-v2025-07_limpio.txt — 120,000 caracteres


## 1) Delimitadores de caso (`CASE n`) y menciones de reglas

**Qué buscamos:** patrones que permitan **segmentar** el Case Book en unidades (un caso = un bloque) y **etiquetar** menciones normativas para metadatos y filtros en recuperación híbrida.

**Ejemplos de patrones** (ajustar según el PDF real):
- `CASE\\s+(\\d+)`: inicio de caso numerado.
- `Rule\\s+(\\d+[a-z]?)` o `RRS\\s+Rule\\s+(\\d+)`: referencias a reglas para extraer lista única por caso.

La celda siguiente cuenta ocurrencias y muestra **ejemplos únicos** para validar a ojo.


In [4]:
RE_CASE = re.compile(r"CASE\s+(\d+)", re.IGNORECASE)
RE_RULE = re.compile(r"(?:Rule|rule)\s+(\d+[a-z]?)", re.IGNORECASE)

cases = RE_CASE.findall(texto_cb)
rules = RE_RULE.findall(texto_cb)

print(f"Marcadores CASE encontrados: {len(cases)} — muestra de números: {sorted(set(cases), key=int)[:20]} ...")
print(f"Referencias Rule n: {len(rules)} — ejemplos únicos: {sorted(set(rules), key=lambda x: int(re.sub(r'[^0-9]', '', x) or 0))[:25]}")

# Primeras coincidencias con contexto corto (para revisar falsos positivos)
for m in RE_CASE.finditer(texto_cb):
    i = m.start()
    ctx = texto_cb[max(0, i - 40) : i + 60].replace("\n", " ")
    print("…", ctx, "…")
    break


Marcadores CASE encontrados: 411 — muestra de números: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '17', '19', '20', '21', '22'] ...
Referencias Rule n: 398 — ejemplos únicos: ['1', '2', '4', '5', '6', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '26', '28', '29', '30', '31', '32']
… , Clear Astern and Clear Ahead; Overlap CASE 12 In determining the right of an inside boat to mark-r …


## 2) Secciones típicas (Facts, Rules, Question, Decision)

Muchos case books usan encabezados de línea. Con regex podés **localizar bloques** antes de pasar a chunking fijo por tamaño: así los embeddings capturan mejor el tipo de contenido y la parte léxica puede hacer match sobre palabras clave dentro del bloque correcto.

Probamos líneas que empiezan por palabras clave (ajustá la lista si tu PDF usa otro formato).


In [5]:
RE_SECCION = re.compile(
    r"^(?:Facts|FACTS|Rules|RULES|Question|QUESTION|Decision|DECISION)\s*[:\s]",
    re.MULTILINE,
)

secciones = RE_SECCION.findall(texto_cb)
from collections import Counter

c = Counter(s.strip().split(":")[0].split()[0] for s in secciones)
print("Encabezados detectados (primer token):", dict(c))


Encabezados detectados (primer token): {}


## 3) Ruido bibliográfico al final de línea / página

**Motivo:** códigos tipo `GBR 1962/25` son poco útiles para consultas en lenguaje natural y pueden **sesgar** embeddings; conviene detectarlos con regex **conservadora** (p. ej. anclada al final del texto de página o del caso) antes de indexar.

Aquí solo **listamos coincidencias candidatas** en la muestra para calibrar el patrón sin borrar nada todavía.


In [6]:
# Patrón ilustrativo: tres letras MAYÚSCULAS + año + / + dígitos
RE_BIB = re.compile(r"\b[A-Z]{3}\s+\d{4}/\d{2,}\b")

bib = RE_BIB.findall(texto_cb)
print(f"Candidatos bibliográficos encontrados: {len(bib)}")
print("Ejemplos:", sorted(set(bib))[:15])


Candidatos bibliográficos encontrados: 1
Ejemplos: ['GBR 1962/25']


## 4) Detección de entidades nombradas (NER)

Además de los patrones **normativos** (`CASE n`, `Rule n`), conviene ver qué **entidades genéricas** aparecen en el texto (personas, organizaciones, lugares, fechas). Eso ayuda a:

- detectar **ruido** indexable (nombres propios poco relevantes para consultas de reglas);
- pensar **metadatos** o filtros (p. ej. eventos, sedes);
- contrastar NER **general** con reglas **regex** del dominio regata.

Usamos [spaCy](https://spacy.io/) con el modelo pequeño en inglés `en_core_web_sm` (rápido en CPU). Instalación una vez por entorno:

```bash
pip install spacy
python -m spacy download en_core_web_sm
```

El análisis se hace sobre un **prefijo** del texto (`MAX_NER_CHARS`) para mantener la celda ágil; aumentá el límite si necesitás cobertura completa.

**Dónde ver el resultado:** la celda de código siguiente muestra tablas y texto en la **salida estándar** de la notebook (debajo de la celda al ejecutarla). Para una vista **con colores** por tipo de entidad, usá la celda **displacy** un poco más abajo.

In [7]:
from collections import Counter, defaultdict

MAX_NER_CHARS = 50_000
fragmento = texto_cb[:MAX_NER_CHARS]

try:
    import spacy
except ImportError as exc:
    raise ImportError(
        "Falta spaCy. En la terminal: pip install spacy && python -m spacy download en_core_web_sm"
    ) from exc

try:
    nlp = spacy.load("en_core_web_sm")
except OSError as exc:
    raise OSError(
        "Falta el modelo en_core_web_sm. Ejecutá: python -m spacy download en_core_web_sm"
    ) from exc

doc = nlp(fragmento)

by_label: dict[str, list[str]] = defaultdict(list)
for ent in doc.ents:
    by_label[ent.label_].append(ent.text)

print(f"Fragmento: {len(fragmento):,} caracteres — tokens spaCy: {len(doc):,}")
print(f"Entidades detectadas: {len(doc.ents):,}\n")

for label in sorted(by_label.keys()):
    texts = by_label[label]
    n_unique = len(set(texts))
    muestra = sorted(set(texts), key=lambda s: (-len(s), s.lower()))[:15]
    print(f"{label:6}  menciones={len(texts):5}  únicas={n_unique:5}  ej.: {muestra}")

print("\n--- Contexto (primeras entidades) ---")
for ent in doc.ents[:12]:
    a = max(0, ent.start_char - 45)
    b = min(len(fragmento), ent.end_char + 45)
    ctx = fragmento[a:b].replace("\n", " ")
    print(f"[{ent.label_:5}] {ent.text!r}  →  …{ctx}…")

# Etiquetas presentes (resumen)
print("\nConteo por etiqueta:", dict(Counter(e.label_ for e in doc.ents)))

Fragmento: 50,000 caracteres — tokens spaCy: 10,291
Entidades detectadas: 526

CARDINAL  menciones=  324  únicas=  102  ej.: ['18.2(a)(1', '18.2(a)(2', '18.2(a', '20.2(c', '43.1(b', '86.1(c', 'three', '16.1', '28.1', '42.1', '69.2', '70.1', '70.2', '88.2', '89.1']
DATE    menciones=   37  únicas=   17  ej.: ['2025-2028', 'each year', 'July 2025', '19.2(b', '20.1(a', '43.1(b', '44.1(b', '2021', '2025', '2028', '14', '21', '25', '33', '35']
EVENT   menciones=    1  únicas=    1  ej.: ['World Sailing Regulations']
FAC     menciones=    1  únicas=    1  ej.: ['Página 5']
GPE     menciones=   34  únicas=    4  ej.: ['Limited', 'London', 'Página', 'UK']
LAW     menciones=    4  únicas=    3  ej.: ['Section 2', 'Rule 13', 'Rule 15']
NORP    menciones=    2  únicas=    1  ej.: ['Finish']
ORDINAL  menciones=   26  únicas=    4  ej.: ['second', 'first', 'third', '4th']
ORG     menciones=   57  únicas=   31  ej.: ['National Authority Abbreviations ARG Federacion', 'Chair World Sailing Racing Rule

### Visualización interactiva de entidades

La celda anterior solo **imprime** listas y fragmentos de contexto. Para ver el mismo análisis **resaltado por tipo** (colores según la etiqueta NER), ejecutá la celda siguiente: spaCy incluye **displacy** y el HTML se muestra **debajo de la celda** en Cursor, VS Code o Jupyter (salida enriquecida). Por rendimiento y legibilidad solo se renderizan los **primeros tokens** (`VIS_MAX_TOKENS`); ajustá ese número si querés ver más texto de una vez.

In [8]:
from spacy import displacy

VIS_MAX_TOKENS = 320
chunk = doc[:VIS_MAX_TOKENS] if len(doc) > VIS_MAX_TOKENS else doc
displacy.render(chunk, style="ent", jupyter=True)

### Resumen de entidades por documento del corpus

Se toma **un documento por cada PDF** en `corpus/`. El texto analizado es `corpus_limpio/<mismo_nombre_stem>_limpio.txt` si existe; si no, la extracción de **todas** las páginas del PDF con `pypdf`. Se usa el mismo modelo `en_core_web_sm` que en la celda anterior.

Los archivos grandes se recorren en ventanas de `CHUNK_CHARS_FOR_NER` caracteres (por `nlp.max_length` y memoria); el total de entidades por documento es la **suma** de menciones en todas las ventanas (cada aparición cuenta una vez).

Al finalizar, se escribe **`corpus_limpio/entidades_ner_corpus.md`**: listado de **todas** las menciones (etiqueta, texto, posición en caracteres) agrupadas por documento, más el resumen tabular en Markdown.

In [9]:
from collections import Counter
from datetime import datetime
from pathlib import Path

import spacy
from pypdf import PdfReader

CHUNK_CHARS_FOR_NER = 900_000
ENTIDADES_NER_MD = CORPUS_LIMPIO_DIR / "entidades_ner_corpus.md"


def texto_desde_pdf_corpus(path: Path) -> str:
    reader = PdfReader(str(path))
    return "\n".join((page.extract_text() or "") for page in reader.pages)


def fuente_y_texto_para_pdf(pdf_path: Path) -> tuple[str, str]:
    limpio = CORPUS_LIMPIO_DIR / f"{pdf_path.stem}_limpio.txt"
    if limpio.is_file():
        return limpio.name, limpio.read_text(encoding="utf-8")
    return pdf_path.name, texto_desde_pdf_corpus(pdf_path)


def ner_texto_completo(
    texto: str, pipe, chunk_chars: int
) -> tuple[int, Counter[str], list[tuple[int, str, str]]]:
    """Conteos, y lista de (posición_inicio_carácter, etiqueta, texto_mención)."""
    by_label: Counter[str] = Counter()
    menciones: list[tuple[int, str, str]] = []
    for start in range(0, len(texto), chunk_chars):
        part = texto[start : start + chunk_chars]
        if len(part) > pipe.max_length:
            pipe.max_length = len(part) + 10_000
        doc_part = pipe(part)
        for e in doc_part.ents:
            by_label[e.label_] += 1
            menciones.append((start + e.start_char, e.label_, e.text))
    menciones.sort(key=lambda t: t[0])
    return len(menciones), by_label, menciones


def md_celda(valor: str) -> str:
    return valor.replace("\n", " ").replace("|", "\\|").strip()


def escribir_entidades_md(
    destino: Path,
    modelo: str,
    filas: list[tuple[str, int, int, Counter[str], list[tuple[int, str, str]]]],
) -> None:
    destino.parent.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().astimezone().strftime("%Y-%m-%d %H:%M %Z")
    lineas: list[str] = [
        "# Entidades NER — corpus completo",
        "",
        f"- **Modelo:** `{modelo}`",
        f"- **Generado:** {ts}",
        f"- **Archivo:** `{destino.name}`",
        "",
        "## Resumen por documento",
        "",
        "| Documento (fuente de texto) | Caracteres | Entidades |",
        "| --- | ---: | ---: |",
    ]
    for fuente, nchars, n_ent, _, _ in filas:
        lineas.append(f"| `{md_celda(fuente)}` | {nchars:,} | {n_ent:,} |")
    total = sum(r[2] for r in filas)
    lineas.append(f"| **TOTAL** | | **{total:,}** |")
    lineas.extend(["", "## Desglose por etiqueta (por documento)", ""])
    for fuente, nchars, n_ent, by_lbl, _ in filas:
        lineas.append(f"### `{md_celda(fuente)}`")
        lineas.append("")
        lineas.append(f"_{n_ent:,} menciones en {nchars:,} caracteres._")
        lineas.append("")
        lineas.append("| Etiqueta | Cantidad |")
        lineas.append("| --- | ---: |")
        for label, k in sorted(by_lbl.items(), key=lambda x: (-x[1], x[0])):
            lineas.append(f"| `{label}` | {k:,} |")
        lineas.append("")
    lineas.extend(["---", "", "## Listado de todas las menciones", ""])
    for fuente, _nchars, n_ent, _by_lbl, menciones in filas:
        lineas.append(f"### `{md_celda(fuente)}`")
        lineas.append("")
        lineas.append("| Nº | Posición (carácter) | Etiqueta | Texto |")
        lineas.append("| --- | ---: | --- | --- |")
        for i, (pos, label, text) in enumerate(menciones, start=1):
            lineas.append(
                f"| {i:,} | {pos:,} | `{md_celda(label)}` | {md_celda(text)} |"
            )
        lineas.append("")
    destino.write_text("\n".join(lineas), encoding="utf-8")


try:
    nlp_corpus = spacy.load("en_core_web_sm")
except OSError as exc:
    raise OSError(
        "Falta el modelo en_core_web_sm. Ejecutá: python -m spacy download en_core_web_sm"
    ) from exc

pdfs = sorted(CORPUS_DIR.glob("*.pdf"))
if not pdfs:
    raise FileNotFoundError("No hay archivos *.pdf en corpus/")

rows: list[tuple[str, int, int, Counter[str], list[tuple[int, str, str]]]] = []
for pdf in pdfs:
    fuente, txt = fuente_y_texto_para_pdf(pdf)
    n_ent, by_lbl, menciones = ner_texto_completo(txt, nlp_corpus, CHUNK_CHARS_FOR_NER)
    rows.append((fuente, len(txt), n_ent, by_lbl, menciones))

escribir_entidades_md(ENTIDADES_NER_MD, "en_core_web_sm", rows)
print(f"Exportado: {ENTIDADES_NER_MD.resolve()}")

w = max(len(r[0]) for r in rows)
hdr = f"{'Documento (fuente de texto)':<{w}}  {'caracteres':>12}  {'entidades':>10}"
print(hdr)
print("-" * len(hdr))
for fuente, nchars, n_ent, _, _ in rows:
    print(f"{fuente:<{w}}  {nchars:>12,}  {n_ent:>10,}")
print("-" * len(hdr))
print(f"{'TOTAL (suma de documentos)':<{w}}  {'':>12}  {sum(r[2] for r in rows):>10,}")

print("\nDesglose por etiqueta NER (por documento):")
for fuente, nchars, n_ent, by_lbl, _ in rows:
    print(f"\n▸ {fuente} — {n_ent:,} entidades en {nchars:,} caracteres")
    for label, k in sorted(by_lbl.items(), key=lambda x: (-x[1], x[0])):
        print(f"    {label:12} {k:6,}")

Exportado: /Users/marcelo.luna/Library/CloudStorage/OneDrive-Personal/Diplomaturas/01. Inteligencia Artificial Aplicada/Materias/05. Taller de Trabajo Final/Repositorio/DIIA_trabajo_final/notebooks auxiliares/corpus_limpio/entidades_ner_corpus.md
Documento (fuente de texto)                              caracteres   entidades
-------------------------------------------------------------------------------
2025-2028-RRS-with-Changes-and-Corrections_limpio.txt       236,114       2,670
The-Call-Book-for-Team-Racing-2025-2028_limpio.txt          123,274       1,277
WS-Case-Book-2025-2028-v2025-07_limpio.txt                  434,601       4,682
-------------------------------------------------------------------------------
TOTAL (suma de documentos)                                                8,629

Desglose por etiqueta NER (por documento):

▸ 2025-2028-RRS-with-Changes-and-Corrections_limpio.txt — 2,670 entidades en 236,114 caracteres
    CARDINAL      1,501
    ORG             321
    

## Cómo llevar esto al pipeline RAG + híbrido

1. **Ingesta:** aplicar limpieza y segmentación validadas aquí → chunks con **metadatos** (`case_id`, `rules_cited`, `section`).
2. **Índice léxico:** mismo texto normalizado que uses en BM25 (tokens alineados con la consulta tras tu tesauro ES↔EN si aplica).
3. **Índice semántico:** embeddings por chunk; filtros opcionales por metadato antes/después del rerank.
4. **Fusión:** combinar scores léxicos y semánticos (p. ej. RRF o pesos), tal como describe la épica de recuperación híbrida en el backlog.

Los patrones de esta notebook son **exploratorios**: cuando los fijes en código, acompañalos con tests/fixtures sobre fragmentos reales del Case Book.
